## 1) Importing libraries

In [13]:
import tensorflow as tf
import matplotlib.pyplot as plt
import os
import pandas as pd
import pathlib
import shutil
import numpy as np

import cv2 as cv

In [14]:
random_state = 1

In [15]:
print(tf.__version__)

2.21.0


## 2) Reading data

### 2.1) Lung opacity

In [16]:
lung_df = pd.read_excel(
    io = "./dataset/COVID.metadata.xlsx"
)

print(lung_df.shape)
lung_df.head(2)

(3616, 4)


,FILE NAME,FORMAT,SIZE,URL
0,COVID-1,PNG,256*256,https://sirm.org/category/senza-categoria/covi...
1,COVID-2,PNG,256*256,https://sirm.org/category/senza-categoria/covi...


### 2.2) Covid

In [17]:
covid_df = pd.read_excel(
    io = "./dataset/Lung_Opacity.metadata.xlsx"
)

print(covid_df.shape)
covid_df.head(2)

(6012, 4)


,FILE NAME,FORMAT,SIZE,URL
0,Lung_Opacity-1,PNG,256*256,https://www.kaggle.com/c/rsna-pneumonia-detect...
1,Lung_Opacity-2,PNG,256*256,https://www.kaggle.com/c/rsna-pneumonia-detect...


### 2.3) Normal

In [18]:
normal_df = pd.read_excel(
    io = "./dataset/Normal.metadata.xlsx"
)

print(normal_df.shape)
normal_df.head(2)

(10192, 4)


,FILE NAME,FORMAT,SIZE,URL
0,NORMAL-1,PNG,256*256,https://www.kaggle.com/c/rsna-pneumonia-detect...
1,NORMAL-2,PNG,256*256,https://www.kaggle.com/c/rsna-pneumonia-detect...


### 2.4) Viral Pneumonia

In [19]:
pneumonia_df = pd.read_excel(
    io = "./dataset/Viral Pneumonia.metadata.xlsx"
)

print(pneumonia_df.shape)
pneumonia_df.head(2)

(1345, 4)


,FILE NAME,FORMAT,SIZE,URL
0,Viral Pneumonia-1,PNG,256*256,https://www.kaggle.com/paultimothymooney/chest...
1,Viral Pneumonia-2,PNG,256*256,https://www.kaggle.com/paultimothymooney/chest...


## 3) Selecting

### 3.1) Training

<p>For training, let's select <code>n = 1000</code> imagens for each category.</p>

In [20]:
n = 1000

train_names = np.concat([
    lung_df.sample(n = n, random_state = random_state)["FILE NAME"].values, 
    covid_df.sample(n = n, random_state = random_state)["FILE NAME"].values,
    normal_df.sample(n = n, random_state = random_state)["FILE NAME"].values,
    pneumonia_df.sample(n = n, random_state = random_state)["FILE NAME"].values
])

### 3.2) Testing

<p align="justify">For testing, let's select new <code>n = 200</code> samples for each category.</p>

In [21]:
n = 200

test_names = np.concat([
    lung_df[~lung_df["FILE NAME"].isin(values = train_names)].sample(n = n, random_state = random_state)["FILE NAME"].values, 
    covid_df[~covid_df["FILE NAME"].isin(values = train_names)].sample(n = n, random_state = random_state)["FILE NAME"].values,
    normal_df[~normal_df["FILE NAME"].isin(values = train_names)].sample(n = n, random_state = random_state)["FILE NAME"].values,
    pneumonia_df[~pneumonia_df["FILE NAME"].isin(values = train_names)].sample(n = n, random_state = random_state)["FILE NAME"].values
])

### 3.3) Validation

<p>At last, for validation, let's select only new <code>n = 100</code> samples for each category.</p>

In [22]:
n = 100

validation_names = np.concat([
    lung_df[~lung_df["FILE NAME"].isin(values = np.concat([train_names, test_names]))].sample(n = n, random_state = random_state)["FILE NAME"].values, 
    covid_df[~covid_df["FILE NAME"].isin(values = np.concat([train_names, test_names]))].sample(n = n, random_state = random_state)["FILE NAME"].values,
    normal_df[~normal_df["FILE NAME"].isin(values = np.concat([train_names, test_names]))].sample(n = n, random_state = random_state)["FILE NAME"].values,
    pneumonia_df[~pneumonia_df["FILE NAME"].isin(values = np.concat([train_names, test_names]))].sample(n = n, random_state = random_state)["FILE NAME"].values
])

## 4) Creating images folder

<p align = "justify">Knowing that the images are all stored in the <code>.dataset/images</code> folder, we will create a <code>chest-xray</code> folder - for example - that will stored all image files separately by category.</p>

### 4.1) Creating folders

In [23]:
if os.path.exists("./chest-xray"):
    shutil.rmtree("./chest-xray")
else:
    print("A pasta não existe.")

# Creating parent folder:
pathlib.Path("./chest-xray").mkdir(exist_ok = True)

# Creating subfolders:
pathlib.Path("./chest-xray/train").mkdir(exist_ok = True)
pathlib.Path("./chest-xray/test").mkdir(exist_ok = True)
pathlib.Path("./chest-xray/val").mkdir(exist_ok = True)


### 4.3) Insert images into subfolders

In [24]:
target_size = (256, 256)

classes = ["Covid", "Lung Opacity", "Normal", "Pneumonia"]

for path in ["./chest-xray/train", "./chest-xray/test", "./chest-xray/val"]:

    for class_name in classes:
        pathlib.Path(f"{path}/{class_name}").mkdir(exist_ok = True)

for subfolder, split_name in [("train", train_names), ("test", test_names), ("val", validation_names)]:
    for path, name in zip(
        np.sort(
            np.array(
                classes*int(len(split_name)/len(classes))
            )
        ) + "/" + split_name
        ,
        split_name
    ):
        
        img = cv.imread(f"./dataset/images/{name}.png", cv.IMREAD_GRAYSCALE)
        img = cv.resize(src = img, dsize = target_size, interpolation = cv.INTER_CUBIC)

        mask = cv.imread(f"./dataset/masks/{name}.png", cv.IMREAD_GRAYSCALE)/255.0

        cv.imwrite(
            filename = f"./chest-xray/{subfolder}/{path}.png", img = img*mask
        )

        # shutil.copy(
        #     src = f"./dataset/images/{name}.png",
        #     dst = f"./chest-xray/{subfolder}/{path}.png"
        # )

In [25]:
img*mask

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(256, 256))